## RTL-SDR: Interaktiver Spektralplot (Marker, Zoom, Pan)

← [RTL-SDR-Samples](rtl_sdr_samples.ipynb)

Dieses Notebook nutzt dieselbe RTL-SDR-Funktionalität wie **rtl_sdr_samples.ipynb**, zeigt das Spektrum aber mit **Markern** und **interaktivem Plot**: Zoom, Pan und Toolbar sind über das **ipympl**-Backend (`%matplotlib widget`) verfügbar. Dafür muss **ipympl** installiert sein: `pip install ipympl`.

### Windows: Treiber-Pfad setzen (vor dem Import)

Falls du unter Windows arbeitest: Zelle ausführen, damit **librtlsdr.dll** aus dem Ordner **rtl-sdr-driver** gefunden wird.

In [1]:
import os
from pathlib import Path

_driver_dir = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    candidate = p / "rtl-sdr-driver"
    if candidate.exists() and (candidate / "librtlsdr.dll").exists():
        _driver_dir = candidate
        break
if _driver_dir is not None:
    _path = str(_driver_dir)
    os.environ["PATH"] = _path + os.pathsep + os.environ.get("PATH", "")
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(_path)
    print("RTL-SDR Treiber gefunden:", _driver_dir)
else:
    print("Hinweis: rtl-sdr-driver (librtlsdr.dll) nicht gefunden.")

RTL-SDR Treiber gefunden: c:\Users\julia\KT-course\rtl-sdr-driver


### Slider (Center Frequency, Gain) und REFRESH-Button

Mit den **Slidern** stellst du Mittenfrequenz (MHz) und Gain (dB) ein; bei Änderung werden neue Samples eingelesen und der Plot aktualisiert. Der Button **REFRESH** liest mit den aktuellen Slider-Werten erneut ein (ohne Slider zu betätigen). **%matplotlib widget** liefert Zoom/Pan in der Toolbar; die Spektralplots verwenden **Marker**.

### Aufgabenstellung (siehe auch in der Funktion `update_plot()`)

**AUFGABE A:**  
N SDR-Samples einlesen. Berechnung der **durchschnittlichen Leistung** der N Samples und der **Peak-Leistung**. Zugrunde liegt eine Normalisierung der Sample-Werte als **Spannungen in Millivolt** an **50 Ω** Lastwiderstand. Ferner: Berechnung der **Gesamtleistung aus dem Spektrum**. Anschließend **Vergleich der gemessenen Leistungen** mit Diskussion des Ergebnisses. Zusätzlich: Berechnung der **Gesamtenergie** der N Samples unter Verwendung der **SI-Einheiten** (Joule = Ws).

**AUFGABE B:**  
Implementiere einen weiteren Slider zur Steuerung einer **Frequenzverschiebung** des eingelesenen Signals. Verwende das **Fourier-Theorem für Frequenzverschiebung** (Hinweis: Multiplikation mit cos + j·sin – von den Studenten zu lösen).

In [2]:
%matplotlib widget

RuntimeError: 'widget' is not a recognised GUI loop or backend name

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Button, VBox, HBox, Label, Output
from IPython.display import display, clear_output

num_samples = 4096
sample_rate_hz = 2.048e6

slider_freq = FloatSlider(value=93.0, min=50.0, max=220.0, step=0.1, description='Center (MHz):',
                          continuous_update=False, readout=True, readout_format='.1f')
slider_gain = FloatSlider(value=20.0, min=0.0, max=49.6, step=0.1, description='Gain (dB):',
                          continuous_update=False, readout=True, readout_format='.1f')
slider_freq_shift = FloatSlider(value=0.0, min=-100.0, max=100.0, step=1.0, description='Freq.-Verschiebung (kHz):',
                                continuous_update=False, readout=True, readout_format='.1f')
btn_refresh = Button(description='REFRESH', button_style='primary', tooltip='Neue Samples mit aktuellen Einstellungen einlesen')
out_err = Output()

# Persistente Figure: einmal erstellen, dann nur Daten aktualisieren – so bleibt Zoom/Pan/Marker erhalten
plt.ioff()
fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=False)
step = max(1, num_samples // 400)
line_i, = axes[0].plot([], [], color='C0', linewidth=0.6, drawstyle='steps-mid')
axes[0].set_ylabel('I (raw int)')
axes[0].set_title('I-Samples (Realteil)')
axes[0].grid(True, alpha=0.3)
line_q, = axes[1].plot([], [], color='C1', linewidth=0.6, drawstyle='steps-mid')
axes[1].set_ylabel('Q (raw int)')
axes[1].set_title('Q-Samples (Imaginärteil)')
axes[1].grid(True, alpha=0.3)
line_mag, = axes[2].plot([], [], color='C2', linewidth=0.6, marker='o', markersize=2, markevery=step)
axes[2].set_xlabel('Frequenz (MHz)')
axes[2].set_ylabel('|FFT|')
axes[2].grid(True, alpha=0.3)
line_dB, = axes[3].plot([], [], color='C3', linewidth=0.6, marker='s', markersize=1.5, markevery=step)
axes[3].set_xlabel('Frequenz (MHz)')
axes[3].set_ylabel('|FFT| (dB)')
axes[3].set_title('Zweiseitiges Spektrum (dB) – mit Markern')
axes[3].grid(True, alpha=0.3)
plt.tight_layout()
plt.ion()

def update_plot(change=None):
    center_freq_mhz = slider_freq.value
    gain_db = slider_gain.value
    f_shift_khz = slider_freq_shift.value
    with out_err:
        clear_output(wait=True)
    try:
        from rtlsdr import RtlSdr
        sdr = RtlSdr()
        sdr.sample_rate = sample_rate_hz
        sdr.center_freq = center_freq_mhz * 1e6
        sdr.gain = gain_db
        samples = sdr.read_samples(num_samples)
        sdr.close()
    except Exception as e:
        with out_err:
            print("RTL-SDR Fehler:", e)
        return
    # AUFGABE B: Frequenzverschiebung des eingelesenen Signals (Fourier-Theorem:
    # Multiplikation mit cos(2*pi*f_shift*t) + j*sin(2*pi*f_shift*t) im Zeitbereich).
    # f_shift = f_shift_khz * 1e3 in Hz; t = np.arange(N) / fs
    # Hier implementieren – danach 'samples' für die Anzeige/FFT verwenden.
    # samples = ...
    I = np.real(samples)
    Q = np.imag(samples)
    I_int = np.clip(np.round(I * 127), -128, 127).astype(np.int32)
    Q_int = np.clip(np.round(Q * 127), -128, 127).astype(np.int32)
    N = len(samples)
    fs = sample_rate_hz
    X = np.fft.fft(samples)
    freq = np.fft.fftfreq(N, 1 / fs)
    magnitude = np.abs(X)
    freq_shifted = np.fft.fftshift(freq)
    magnitude_shifted = np.fft.fftshift(magnitude)
    magnitude_dB = 20 * np.log10(magnitude_shifted + 1e-12)
    freq_mhz = freq_shifted / 1e6
    line_i.set_data(np.arange(N), I_int)
    line_q.set_data(np.arange(N), Q_int)
    line_mag.set_data(freq_mhz, magnitude_shifted)
    line_dB.set_data(freq_mhz, magnitude_dB)
    axes[2].set_title(f'Zweiseitiges Spektrum @ {center_freq_mhz:.1f} MHz, Gain {gain_db:.1f} dB')
    # --- AUFGABE A: Aufgabenstellung (Berechnungen hier bzw. in out_err ausgeben) ---
    # N SDR-Samples einlesen (erledigt). Berechnung der durchschnittlichen Leistung der N Samples,
    # Berechnung der Peak-Leistung. Zugrunde liegt Normalisierung der Sample-Werte als Spannungen
    # in Millivolt an 50-Ohm-Lastwiderstand. Leistungen in linearer Einheit (W) und in dBm (Referenz 1 mW) angeben.
    # Ferner: Berechnung der Gesamtleistung aus dem Spektrum (W und dBm).
    # Anschließend Vergleich der gemessenen Leistungen mit Diskussion des Ergebnisses.
    # Nächste Aufgabe: Berechnung der Gesamtenergie der N Samples unter Verwendung der SI-Einheiten (Joule = Ws).
    # P_avg = ...  # mittlere Leistung (Zeitbereich), dann P_avg_dBm = 10*log10(P_avg/1e-3)
    # P_peak = ...  # Peak-Leistung, dann P_peak_dBm
    # P_spektrum = ...  # Gesamtleistung aus Spektrum (Parseval), dann P_spektrum_dBm
    # E_gesamt = ...  # Gesamtenergie in Joule
    for ax in axes:
        ax.relim()
        ax.autoscale_view()
    fig.canvas.draw_idle()

slider_freq.observe(update_plot, names='value')
slider_gain.observe(update_plot, names='value')
slider_freq_shift.observe(update_plot, names='value')
btn_refresh.on_click(lambda _: update_plot())
display(VBox([Label('Center (MHz), Gain (dB), Frequenzverschiebung (kHz); REFRESH = neue Samples'),
              HBox([slider_freq, slider_gain, slider_freq_shift, btn_refresh]), out_err, fig.canvas]))
update_plot()

ModuleNotFoundError: No module named 'ipywidgets'

→ [RTL-SDR Interaktiv](../02_02_SDR_Intro/rtl_sdr_spectrum_interactive.ipynb) | Lösung: [02_03_LABOR-1-solution](02_03_LABOR-1-solution.ipynb)